In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Konfigurace zmírňování chyb

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Beta verze nového modelu spouštění je nyní dostupná. Řízený model spouštění poskytuje větší flexibilitu při přizpůsobování pracovního postupu zmírňování chyb. Více informací najdeš v průvodci [Řízený model spouštění](/guides/directed-execution-model).


<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Techniky zmírňování chyb umožňují uživatelům snižovat chyby v Circuit modelováním šumu zařízení v době spouštění. To obvykle
vede k režijním nákladům na kvantové předzpracování spojeným s trénováním modelu a
na klasické následné zpracování za účelem zmírnění chyb v surových výsledcích
pomocí vygenerovaného modelu.

Primitiv Estimator podporuje několik technik zmírňování chyb, včetně [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) a [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Vysvětlení každé z nich najdeš v části [Techniky zmírňování a potlačování chyb](error-mitigation-and-suppression-techniques). Při použití primitivů můžeš jednotlivé metody zapínat nebo vypínat. Podrobnosti najdeš v sekci [Vlastní nastavení chyb](#advanced-error).

> **Note:** Sampler nepodporuje zmírňování chyb, ale pro lokální zmírňování chyb můžeš použít balíček [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (bezmapové zmírňování chyb při měření).

Estimator také podporuje `resilience_level`. Úroveň odolnosti určuje, jak moc odolnost vůči
chybám vybudovat. Vyšší úrovně generují přesnější výsledky, ale za cenu
delší doby zpracování. Úrovně odolnosti lze použít ke konfiguraci
kompromisu mezi náklady a přesností při aplikaci zmírňování chyb na tvůj primitivní
dotaz. Zmírňování chyb snižuje chyby (zkreslení) ve výsledcích zpracováním
výstupů z kolekce nebo souboru příbuzných Circuit. Míra snížení chyb závisí na použité metodě. Úroveň odolnosti abstrahuje podrobnou volbu metody zmírňování chyb, aby uživatelé mohli uvažovat o kompromisu mezi náklady a přesností vhodném pro jejich aplikaci.

S ohledem na to každá úroveň odpovídá metodě nebo metodám s
rostoucí úrovní režijních nákladů na kvantové vzorkování, což ti umožňuje experimentovat
s různými kompromisy mezi časem a přesností. Následující tabulka ukazuje,
které úrovně a odpovídající metody jsou dostupné pro každý z primitivů.

> **Info:** Zmírňování chyb je specifické pro úlohu, takže techniky, které můžeš
> použít, se liší podle toho, zda vzorkuješ distribuci nebo generuješ
> očekávané hodnoty.

<span id="resilience-table"></span>

Estimator podporuje následující úrovně odolnosti. Sampler úrovně odolnosti nepodporuje.

| Úroveň odolnosti | Definice                                                                                                      | Technika                                                                  |
|------------------|---------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Bez zmírňování                                                                                                | Žádná                                                                     |
| 1 [Výchozí]      | Minimální náklady na zmírňování: Zmírňování chyb spojených s chybami při čtení                               | Twirled Readout Error eXtinction (TREX) a twirling při měření             |
| 2                | Střední náklady na zmírňování. Typicky snižuje zkreslení v estimátorech, ale není zaručeno, že bude nulové.  | Úroveň 1 + Zero Noise Extrapolation (ZNE) a Gate twirling

> **Info:** Úrovně odolnosti jsou momentálně v beta verzi, takže režijní náklady na vzorkování a
> kvalita řešení se budou lišit Circuit od Circuit. Nové funkce,
> pokročilé možnosti a nástroje pro správu budou vydávány průběžně.
> Konkrétní metody zmírňování chyb nejsou zaručeny pro každou úroveň odolnosti.

## Konfigurace Estimatoru s úrovněmi odolnosti
Úrovně odolnosti můžeš použít k určení technik zmírňování chyb, nebo můžeš nastavit vlastní techniky jednotlivě, jak je popsáno v části [Vlastní nastavení chyb.](#advanced-error)

<details>
<summary>Úroveň odolnosti 0</summary>

Na uživatelský program se neaplikuje žádné zmírňování chyb.

</details>

<details>
<summary>Úroveň odolnosti 1</summary>

Úroveň 1 aplikuje **zmírňování chyb při čtení** a **twirling při měření** pomocí bezmapové techniky známé
jako Twirled Readout Error eXtinction (TREX). Snižuje chybu při měření
diagonalizací šumového kanálu spojeného s měřením náhodným
překlápěním Qubitů pomocí X Gate těsně před měřením.
Škálovací člen z diagonálního šumového kanálu je získán
benchmarkingem náhodných Circuit inicializovaných v nulovém stavu. To umožňuje
službě odstranit zkreslení z očekávaných hodnot vyplývající z
šumu při čtení. Tento přístup je podrobněji popsán v článku [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Úroveň odolnosti 2</summary>

Úroveň 2 aplikuje **techniky zmírňování chyb zahrnuté v úrovni 1** a navíc aplikuje **Gate twirling** a používá **metodu Zero Noise Extrapolation (ZNE)**. ZNE vypočítá
očekávanou hodnotu pozorovatelné pro různé faktory šumu
(fáze amplifikace) a poté pomocí naměřených očekávaných hodnot
odvodí ideální očekávanou hodnotu při nulovém šumu (fáze
extrapolace). Tento přístup má tendenci snižovat chyby v očekávaných hodnotách, ale
není zaručeno, že produkuje nestranný výsledek.

![Tento obrázek ukazuje graf. Osa x je označena Faktor amplifikace šumu. Osa y je označena Očekávaná hodnota. Vzestupně stoupající čára je označena Zmírněná hodnota. Body poblíž čáry jsou hodnoty zesílené šumem. Je zde vodorovná čára těsně nad osou X označená Přesná hodnota.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustrace metody ZNE")

Režijní náklady této metody rostou s počtem faktorů šumu.
Výchozí nastavení vzorkuje očekávanou hodnotu při třech faktorech šumu,
což vede k přibližně 3násobnému nárůstu nákladů při použití této úrovně odolnosti.

Na úrovni 2 metoda TREX náhodně překlápí Qubity pomocí X Gate těsně před měřením
a překlápí odpovídající naměřený bit, pokud byl aplikován X Gate. Tento přístup je podrobněji popsán v článku [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Příklad
Rozhraní `EstimatorV2` umožňuje uživatelům bezproblémově pracovat s různými
metodami zmírňování chyb ke snížení chyb v očekávaných hodnotách
pozorovatelných. Následující kód používá Zero Noise Extrapolation a zmírňování chyb při čtení pouhým
nastavením `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Vlastní nastavení chyb
Můžeš zapínat a vypínat jednotlivé metody zmírňování a potlačování chyb, včetně dynamického oddělování, Gate a měřícího twirling, zmírňování chyb při měření, PEC a ZNE. Vysvětlení každé z nich najdeš v části [Techniky zmírňování a potlačování chyb](error-mitigation-and-suppression-techniques).

> **Note:** - Ne všechny možnosti jsou dostupné pro oba primitivy. Viz [tabulka dostupných možností](runtime-options-overview#options-table) pro seznam dostupných možností.
> - Ne všechny metody fungují společně na všech typech Circuit. Viz [tabulka kompatibility funkcí](runtime-options-overview#options-compatibility-table) pro podrobnosti.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">